# NumPyro Mixture Model

This notebook mirrors the logic in `mcmc_mixture.ipynb` using NumPyro:

- `w ~ Dirichlet([1, 1])`
- `theta_1 ~ Uniform(-1, 0)`
- `theta_2 ~ Uniform(2, 3)`
- for each observation: `z ~ Categorical(w)`, `x_1 ~ Normal(theta_1, 1.0)`, `x_2 ~ Normal(theta_2, 1.5)`, `x = select([x_1, x_2], z)`, `obs ~ Normal(x, 0.25)`

Inference is done with `DiscreteHMCGibbs(NUTS(...))` so the discrete assignments `z` are sampled explicitly while the continuous variables are handled by NUTS.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import jax
import jax.numpy as jnp
from jax import random

import numpyro
import numpyro.distributions as dist
from numpyro import handlers
from numpyro.infer import MCMC, NUTS, DiscreteHMCGibbs

NUM_WANTED_CHAINS = 4

# try:
#     GPU_AVAILABLE = len(jax.devices("gpu")) > 0
# except RuntimeError:
#     GPU_AVAILABLE = False

GPU_AVAILABLE = False

if GPU_AVAILABLE:
    numpyro.set_platform("gpu")
    CHAIN_METHOD = "vectorized"
    print("Using GPU with vectorized chains.")
else:
    numpyro.set_platform("cpu")
    numpyro.set_host_device_count(NUM_WANTED_CHAINS)
    CHAIN_METHOD = "parallel"
    print("Using CPU with parallel chains.")

In [ ]:
N = 1000
NUM_WARMUP = 2000
NUM_SAMPLES = 2000
NUM_CHAINS = NUM_WANTED_CHAINS

rng_key = random.PRNGKey(0)
print("jax.default_backend()", jax.default_backend())
print("jax.local_device_count()", jax.local_device_count())
print("jax.devices()", jax.devices())

In [ ]:
def mixture_model(obs=None, N=1000):
    K = 2

    w = numpyro.sample("w", dist.Dirichlet(jnp.ones(K)))
    theta_1 = numpyro.sample("theta_1", dist.Uniform(-1.0, 0.0))
    theta_2 = numpyro.sample("theta_2", dist.Uniform(2.0, 3.0))

    with numpyro.plate("n", N):
        z = numpyro.sample("z", dist.Categorical(probs=w))
        x_1 = numpyro.sample("x_1", dist.Normal(theta_1, 1.0))
        x_2 = numpyro.sample("x_2", dist.Normal(theta_2, 1.5))
        x = jnp.where(z == 0, x_1, x_2)
        numpyro.deterministic("x", x)
        numpyro.sample("obs", dist.Normal(x, 0.25), obs=obs)

In [ ]:
prior_key, mcmc_key = random.split(rng_key)
prior_trace = handlers.trace(handlers.seed(mixture_model, prior_key)).get_trace(N=N)

init_state = {
    name: site["value"]
    for name, site in prior_trace.items()
    if site["type"] in {"sample", "deterministic"}
}
observed = init_state["obs"]

print("w", init_state["w"])
print("theta_1", init_state["theta_1"])
print("theta_2", init_state["theta_2"])
print("obs shape", observed.shape)

In [ ]:
obs_np = np.asarray(observed)
z_np = np.asarray(init_state["z"])

theta_1_true = float(init_state["theta_1"])
theta_2_true = float(init_state["theta_2"])
w_true = np.asarray(init_state["w"])

plt.figure(figsize=(10, 5))
plt.hist(obs_np, bins=30, density=True, histtype="step", label="Observed")
plt.hist(obs_np[z_np == 0], bins=30, density=True, histtype="step", linestyle="--", color="red", label="Component 1")
plt.hist(obs_np[z_np == 1], bins=30, density=True, histtype="step", linestyle="--", color="blue", label="Component 2")

xx = np.linspace(-5.0, 10.0, 300)
pdf_0 = w_true[0] * np.exp(dist.Normal(theta_1_true, 1.0).log_prob(jnp.asarray(xx)))
pdf_1 = w_true[1] * np.exp(dist.Normal(theta_2_true, 1.5).log_prob(jnp.asarray(xx)))
plt.plot(xx, np.asarray(pdf_0), "r--", label="Component 1 PDF")
plt.plot(xx, np.asarray(pdf_1), "b--", label="Component 2 PDF")
plt.plot(xx, np.asarray(pdf_0 + pdf_1), "k--", label="Mixture PDF")
plt.axvline(theta_1_true, color="red", linestyle=":", label="True theta_1")
plt.axvline(theta_2_true, color="blue", linestyle=":", label="True theta_2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
kernel = DiscreteHMCGibbs(NUTS(mixture_model))
mcmc = MCMC(
    kernel,
    num_warmup=NUM_WARMUP,
    num_samples=NUM_SAMPLES,
    num_chains=NUM_CHAINS,
    chain_method=CHAIN_METHOD,
    progress_bar=True,
)

mcmc.run(mcmc_key, obs=observed, N=N)
mcmc.print_summary()

In [ ]:
samples = mcmc.get_samples(group_by_chain=True)
posterior = {name: np.asarray(value) for name, value in samples.items()}

print("theta_1 true", theta_1_true)
print("theta_1 mean", posterior["theta_1"].mean())
print("theta_2 true", theta_2_true)
print("theta_2 mean", posterior["theta_2"].mean())
print("w true", w_true)
print("w mean", posterior["w"].mean(axis=(0, 1)))

In [ ]:
plt.figure(figsize=(10, 4))
for chain in range(posterior["theta_1"].shape[0]):
    plt.plot(posterior["theta_1"][chain], alpha=0.5)
plt.axhline(theta_1_true, color="red", linestyle="--", label="True theta_1")
plt.title("Trace plot for theta_1")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
for chain in range(posterior["theta_2"].shape[0]):
    plt.plot(posterior["theta_2"][chain], alpha=0.5)
plt.axhline(theta_2_true, color="blue", linestyle="--", label="True theta_2")
plt.title("Trace plot for theta_2")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
theta_1_fit = posterior["theta_1"].mean()
theta_2_fit = posterior["theta_2"].mean()
w_fit = posterior["w"].mean(axis=(0, 1))

plt.figure(figsize=(10, 5))
plt.hist(obs_np, bins=30, density=True, histtype="step", label="Observed")

pdf_0_fit = w_fit[0] * np.exp(dist.Normal(theta_1_fit, 1.0).log_prob(jnp.asarray(xx)))
pdf_1_fit = w_fit[1] * np.exp(dist.Normal(theta_2_fit, 1.5).log_prob(jnp.asarray(xx)))

plt.plot(xx, np.asarray(pdf_0_fit), "r-", label="Fitted component 1 PDF")
plt.plot(xx, np.asarray(pdf_1_fit), "b-", label="Fitted component 2 PDF")
plt.plot(xx, np.asarray(pdf_0_fit + pdf_1_fit), "k-", label="Fitted mixture PDF")
plt.axvline(theta_1_true, color="red", linestyle="--", label="True theta_1")
plt.axvline(theta_2_true, color="blue", linestyle="--", label="True theta_2")
plt.axvline(theta_1_fit, color="red", linestyle=":", label="Posterior mean theta_1")
plt.axvline(theta_2_fit, color="blue", linestyle=":", label="Posterior mean theta_2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
z_samples = posterior["z"]
p_component_0 = (z_samples == 0).mean(axis=(0, 1))
p_component_1 = (z_samples == 1).mean(axis=(0, 1))

plt.figure(figsize=(10, 5))
plt.scatter(obs_np, p_component_0, s=6, alpha=0.5, label="P(z = 0 | obs)")
plt.scatter(obs_np, p_component_1, s=6, alpha=0.5, label="P(z = 1 | obs)")
plt.xlabel("Observed value")
plt.ylabel("Posterior assignment probability")
plt.grid(alpha=0.3)
plt.legend()
plt.show()